# Stage 3 - Rerieval Pipeline

## Data
2_chunks.json
1_queries.json

## Install
```
pip install sentence_transformers numpy 
pip install tqdm ipywidgets
pip install chromadb
pip install openai
```

## cuda driver
sudo apt-get update
sudo apt-get install -y ubuntu-drivers-common
sudo ubuntu-drivers autoinstall

sudo reboot


# docker
sudo docker run -d --gpus all -p 8080:8080 --name llm ghcr.io/ggml-org/llama.cpp:server-cuda \
  -hf unsloth/Qwen3.5-4B-GGUF:Q4_K_M --host 0.0.0.0 --port 8080 -ngl 99 -c 4096

sudo docker start llm

curl http://localhost:8080/health 



In [1]:
'''
1. Load chunks
'''
import json

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

texts = [c["text"] for c in chunks]
metas = [{k: c[k] for k in ("id", "doc_id", "category", "format", "title")}
         for c in chunks]

print("Разделов:", len(chunks))

Разделов: 176


In [3]:
'''
2. Embed chunks
'''
import numpy as np
from sentence_transformers import SentenceTransformer

def embed_passages(model, texts):
    """Эмбеддинги документов через multilingual-e5: префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True, batch_size=64)


# --- Считаем эмбеддинги корпуса (e5-base, 768 координат) и сохраняем ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")
corpus_vecs = np.ascontiguousarray(embed_passages(embedder, texts), dtype="float32")
np.save("tmp/kb_vecs.npy", corpus_vecs)   # Эмбеддинги — в файл, переиспользуем

print(f"Чанков: {len(texts)}, эмбеддинги: {corpus_vecs.shape}")
print("Пример метаданных:", metas[1])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18784.41it/s]


Чанков: 176, эмбеддинги: (176, 768)
Пример метаданных: {'id': 1, 'doc_id': 1, 'category': 'Функции и контент', 'format': 'text', 'title': 'Синхронизация между устройствами'}


In [4]:
'''
3. Load vectors to DB
'''
import chromadb
from sentence_transformers import SentenceTransformer

# --- Создаём локальную коллекцию Chroma ---
client = chromadb.PersistentClient(path="chroma_kb")     # Хранение на диске
try:
    client.delete_collection("knowledge_base")           # Чистим прошлый прогон
except Exception:
    pass

col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}})         # Косинусная близость

col.add(
    ids=[str(i) for i in range(len(texts))],             # Уникальные id
    embeddings=corpus_vecs.tolist(),                     # Векторы чанков
    documents=texts,                                     # Сами тексты
    metadatas=metas,                                     # Метаданные
)
print("Записей в коллекции:", col.count())

Записей в коллекции: 176


In [4]:
'''
Example:
Семантический поиск с фильтром по категории
'''

QUERY = "какая версия webOS или Tizen нужна для приложения на телевизоре"

e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=5,
  where={"category": "Устройства и качество"}
)                   # Фильтр по метаданным

print(f"\nЗапрос: {QUERY!r} | фильтр category='Устройства и качество'")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] [{m['category']}] {doc}")

# Для сравнения — тот же запрос без фильтра
res2 = col.query(
  query_embeddings=[q_vec.tolist()], 
  n_results=5
)
print("\nБез фильтра категории первых трёх:",
      [m["category"] for m in res2["metadatas"][0]])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4873.96it/s]



Запрос: 'какая версия webOS или Tizen нужна для приложения на телевизоре' | фильтр category='Устройства и качество'
  dist=0.129 [id: 141] [doc_id: 146] [title: На каких устройствах работает «Фильмоскоп»] [Устройства и качество] # На каких устройствах работает «Фильмоскоп» Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года), LG (webOS 4.0 и новее), телевизоры и приставки на Android TV 9 и новее. ТВ-приставки: Android TV, Apple TV (tvOS 15 и новее), приставки операторов-партнёров. Телефоны и планшеты: iOS 15 и новее, Android 9 и новее. Компьютеры: браузеры Chrome, Firefox, Safari и Яндекс Браузер актуальных версий — без установки приложения. Игровые консоли: PlayStation 5 и Xbox Series. На устройствах вне этого списка работа приложения не гарантируется, даже если его удалось установить.
  dist=0.139 [id: 80] [doc_id: 86] [title: Поддерживаемые устройства] [Устройства и качество] # Поддерживаемые устройства На каких устройствах работает "Фильмоскоп" Телевизоры Smart TV: Samsung (Ti

In [5]:
'''
Test: Semantic search
Correct doc: #140
'''
QUERY = "боюсь, что ребёнок случайно купит фильм с моей карты"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=5,
)           

print(f"\nЗапрос: {QUERY}")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] {doc}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3065.27it/s]



Запрос: боюсь, что ребёнок случайно купит фильм с моей карты
  dist=0.161 [id: 164] [doc_id: 171] [title: Детский контент и возрастные рейтинги] Возрастной рейтинг в карточке фильма — не оценка редакции и не жанр, а формальная маркировка, с которой картина вышла в прокат. Сервис её показывает, но не назначает. Присваивают рейтинг при выдаче прокатного удостоверения, и относится он к фильму целиком. Из-за этого мультфильм, который выглядит совсем детским, иногда стоит на ступень выше ожидаемого: хватает одной тревожной сцены или одной реплики, чтобы категория поднялась. Обратное тоже бывает — 0+ не означает «для малышей», под этой маркировкой выходит и взрослое кино, в котором просто нет ничего запрещённого. Иностранные мультфильмы приходят со своей маркировкой, и она не совпадает с российской: при выходе в наш прокат категорию присваивают заново. Полностью на рейтинг полагаться не стоит. Он говорит о том, что в картине нет запрещённого для этого возраста, но ничего не говорит о темпе,

In [5]:
'''
4. Retrieve func
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.83

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

def retrieve(query, col=col, embedder=embedder, k=5, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3382.74it/s]


In [6]:
'''
4. Retrieve dict
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.83

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

def retrieve_dict(query, col=col, embedder=embedder, k=5, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3374.65it/s]


In [7]:
'''
4.1 Test: all queries (not gap)
'''
import json

# load queries
with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]


for qi, q in enumerate(queries):
  hits = retrieve(q["query"])
  print(f"\n[{qi}] {q['query']}")
  if not hits:
    print("    (ничего выше порога — пайплайн вернул пусто)")
  for cid, doc, sim, id in hits:
    print(f"    sim={sim:.3f} #{cid} {doc[:58]} #{id}")


[0] что означает ошибка P-102
    sim=0.861 #85 # Ошибка P-102: превышен лимит просмотров Ошибка P-102 поя #78
    sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает св #172
    sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить #129
    sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает  #130
    sim=0.856 #129 Ошибка P-105 появляется, когда на устройстве установлена у #122

[1] какая версия webOS или Tizen нужна для приложения на телевизоре
    sim=0.871 #146 # На каких устройствах работает «Фильмоскоп» Телевизоры Sm #140
    sim=0.861 #86 # Поддерживаемые устройства На каких устройствах работает  #79
    sim=0.839 #139 Если вы не знаете, к какой платформе относится ваш телевиз #131
    sim=0.838 #129 Ошибка P-105 появляется, когда на устройстве установлена у #122
    sim=0.835 #172 Приложение на телевизоре обновляется само, если в магазине #165

[2] обязателен ли HDCP 2.2 для просмотра в 4K
    sim=0.874 #110 Что нужн

In [12]:
'''
4.2 Test: compare with relevant docs
'''
# precision@K, recall@K и MRR

relevant = {qi: set(q["relevant_docs"]) for qi, q in enumerate(queries)}

problem = []
for qi, q in enumerate(queries):
    hits = retrieve(q["query"])
    # hits = retrieve(q["query"], k=5, threshold=0.0)
    top_ids = [int(cid) for cid, _, _, id in hits]
    rel = relevant[qi]
    is_problem = (not top_ids) or (top_ids[0] not in rel)
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    hits = retrieve(q["query"], col, embedder, k=10, threshold=0.0)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for cid, doc, sim, id in hits:
        mark = "+" if int(cid) in rel else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}")

print
print("\nЗапрос [8] без порога:",
  [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.0)])
print("Запрос [8] с порогом 0.83:",
  [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.83)])

Проблемные запросы: [3, 10, 15, 18, 19, 20, 22, 23, 24, 28, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 46, 47, 48, 49, 50, 51, 53, 55]

[3] как подключить саундбар по HDMI ARC  (релевантные id: [22])
      sim=0.842 #125 Если при воспроизведении звук идет, а изображения нет — экра
      sim=0.834 #69 Объёмный звук дома собирают двумя способами, и выбор между н
      sim=0.825 #75 {"service": "player-api", "version": "2.14.3", "env": "produ
      sim=0.814 #50 Встроенные динамики телевизора делают тонкими вместе с корпу
      sim=0.814 #150 # Игры и кино на одной консоли У вас уже есть всё для домашн
    + sim=0.813 #22 Обозначение 5.1 описывает раскладку каналов: пять основных к
      sim=0.809 #112 Если изображение идёт без звука или звук отстаёт от картинки
      sim=0.803 #110 Что нужно для просмотра в 4K HDR — Справка «Фильмоскопа» Гла
      sim=0.803 #130 Настройки профиля открываются с экрана выбора профиля или из
      sim=0.801 #60 Стабильность просмотра сильно зависит от того

In [8]:
'''
5. LLM for Expansion & HyDE
'''
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [ ]:
'''
6. Expansion and Ranking
'''
from collections import defaultdict

# def search(query, n=50):
#     """Поиск по тексту: кодирует его как query: и возвращает индексы
#     top-n документов по убыванию косинусного сходства."""
#     ret = retrieve(q["query"], col, embedder, k=5, threshold=0.0)
#     return ret


def expand(query, n=5):
  """Query expansion: n перефразировок запроса официальными словами базы."""
  text = generate([
      {"role": "system", "content": "Ты помощник поиска по базе знаний "
                                    "онлайн-кинотеатра."},
      {"role": "user", "content": f"Переформулируй запрос клиента {n} способами "
        f"официальными словами из базы знаний. Используй полные фразы (не одно "
        f"слово), синонимы и термины. По одной формулировке на строку, без "
        f"нумерации.\n\nЗапрос: {query}"}])
  expanded = [l.strip() for l in text.splitlines() if l.strip()][:n]
  # print('\n>>> expand: ', query, '\n', expanded)
  return expanded


def rrf(ranked_lists, k=60, top_k=5):
  """Reciprocal Rank Fusion: сливаем несколько выдач по рангам."""
  scores = defaultdict(float)
  for lst in ranked_lists:
      for rank, doc in enumerate(lst, start=1):
          scores[doc] += 1.0 / (k + rank)
  ranked = sorted(scores.items(), key=lambda x: -x[1])
  return [doc for doc, _ in ranked[:top_k]]

In [15]:
query = queries[30]['query']
rel = set(queries[30]["relevant_docs"])

# basic search
basic = retrieve(query, k=5)
print('>>> query: ', query)
for i in basic:
  print('>>> basic: ', i)

paras = expand(query, n=10)
for i in paras:   
  print('>>> paras: ', i)

# expanded search
for p in paras:
  found = retrieve(p, k=5) 
  print('>>> found: ', found)

# qs = [basic]
# for item in found:
#   if item not in qs:
#       qs.append(item)
# expanded = rrf(qs)

# for item in expanded:
#   print('>>> expanded: ', expanded[:5])


# mark = lambda i: "+" if i in rel else " "
# print(f"\n[{qi}] {query['query']}")

# for p in paras:
#   print("   ·", p)

# print("  обычный поиск top5:",
#   [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in basic[:5]])
# print("  query expansion top5:",
#   [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in expanded[:5]])

>>> query:  боюсь, что ребёнок случайно купит фильм с моей карты
>>> basic:  (171, 'Возрастной рейтинг в карточке фильма — не оценка редакции и не жанр, а формальная маркировка, с которой картина вышла в прокат. Сервис её показывает, но не назначает. Присваивают рейтинг при выдаче прокатного удостоверения, и относится он к фильму целиком. Из-за этого мультфильм, который выглядит совсем детским, иногда стоит на ступень выше ожидаемого: хватает одной тревожной сцены или одной реплики, чтобы категория поднялась. Обратное тоже бывает — 0+ не означает «для малышей», под этой маркировкой выходит и взрослое кино, в котором просто нет ничего запрещённого. Иностранные мультфильмы приходят со своей маркировкой, и она не совпадает с российской: при выходе в наш прокат категорию присваивают заново. Полностью на рейтинг полагаться не стоит. Он говорит о том, что в картине нет запрещённого для этого возраста, но ничего не говорит о темпе, громкости и о том, испугается ли ваш ребёнок конкретной сцены

In [10]:
'''
6.1 Test expansion on 3 queries
'''

# for qi in [30]:
for qi in [22, 30, 33]:
  query = queries[qi]
  rel = set(queries[qi]["relevant_docs"])

  # basic search
  basic = retrieve(query['query'], k=5)
  # print('>>> basic: ', basic)

  paras = expand(query['query'])
  # paras.append(query['query'])
  # print('>>> paras: ', paras)

  # expanded search
  found = [retrieve(p, k=5) for p in paras]
  # print('>>> found: ', found)

  qs = [basic]
  for item in found:
    if item not in qs:
        qs.append(item)
  expanded = rrf(qs)

  # for item in expanded:
  #   print('>>> expanded: ', expanded[:5])

  
  mark = lambda i: "+" if i in rel else " "
  print(f"\n[{qi}] {query['query']}")
  
  for p in paras:
    print("   ·", p)

  print("  обычный поиск top5:",
    [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in basic[:5]])
  print("  query expansion top5:",
    [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in expanded[:5]])


[22] начал в метро с телефона, дома хочу досмотреть на большом экране с той же секунды
   · Запрос на воспроизведение фильма с точки останова с мобильного устройства на домашнем телевизоре.
   · Начало просмотра с мобильного телефона и продолжение на большом экране в домашнем кинотеатре.
   · Переход просмотра с мобильного устройства на домашний телевизор с сохранением текущего времени.
   · Запуск фильма с мобильного телефона и продолжение просмотра на большом экране дома.
   · Воспроизведение контента с точки останова с мобильного устройства на домашнем телевизоре.
  обычный поиск top5: [' #24 # Какой тариф подойдёт именно ва', ' #39 Кино давно смотрят не с одного э', ' #138 Ограничения: **скачивать можно н', ' #101 Офлайн-просмотр — это фильмы и с']
  query expansion top5: [' #24 # Какой тариф подойдёт именно ва', ' #24 # Какой тариф подойдёт именно ва', ' #165 Приложение на телевизоре обновля', ' #39 Кино давно смотрят не с одного э', '+#104 Позиция просмотра сохраняется ав']

[30

In [11]:
def retrieve_hyde(query, answer, col, embedder, k=5, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out


def retrieve_hyde_dict(query, answer, k=5, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    ) 
  return out


def hyde(query, top_k=5):
    """HyDE: ищем по среднему векторов запроса и гипотетического ответа."""
    doc = generate([
        {"role": "system", "content": "Ты пишешь короткие ответы из базы знаний "
                                      "онлайн-кинотеатра."},
        {"role": "user", "content": f"Напиши короткий (1-2 предложения) фрагмент-"
         f"ответ из базы знаний на вопрос клиента официальным языком. Только "
         f"текст ответа.\n\nВопрос: {query}"}], max_tokens=110)
    hits = retrieve_hyde(query, doc, col, embedder, top_k)
    return doc, hits


for qi in [22, 30, 33]:
    query = queries[qi]["query"]
    rel = set(queries[qi]["relevant_docs"])
    doc, hits = hyde(query)
    mark = lambda i: "+" if i in rel else " "

    found = retrieve(query)
    # print('>>> found: ', found)

    print(f"\n[{qi}] {query}")
    print("  гипотетический ответ:", doc)
    print("  обычный поиск top5:",
        [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in found[:5]])
    print("  HyDE top5:", [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in hits[:5]]) 


[22] начал в метро с телефона, дома хочу досмотреть на большом экране с той же секунды
  гипотетический ответ: Для просмотра фильма с того же места, где вы начали его в метро, необходимо создать аккаунт и авторизоваться на устройстве дома. После входа в систему вы сможете продолжить просмотр с точки, где прервали сеанс.
  обычный поиск top5: [' #24 # Какой тариф подойдёт именно ва', ' #39 Кино давно смотрят не с одного э', ' #138 Ограничения: **скачивать можно н', ' #101 Офлайн-просмотр — это фильмы и с']
  HyDE top5: [' #24 # Какой тариф подойдёт именно ва', ' #90 Совместный просмотр — Справка «Ф', ' #138 Ограничения: **скачивать можно н', ' #57 # Профили в аккаунте Профиль — э', ' #39 Кино давно смотрят не с одного э']

[30] боюсь, что ребёнок случайно купит фильм с моей карты
  гипотетический ответ: Для предотвращения случайных покупок фильмов на вашей карте необходимо отключить функцию «Автопрокат» в настройках аккаунта или временно заблокировать доступ к онлайн-кинотеатру.
  обыч

In [12]:
import numpy as np

def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0

rows = {"Обычный поиск": [], "Query expansion": [], "HyDE": []}

for q in queries:
    query, rel = q["query"], set(q["relevant_docs"])

    base = retrieve(query)
    base_docs = [t[0] for t in base]
    rows["Обычный поиск"].append(recall_at_k(base_docs, rel))

    exp = rrf([retrieve(query)] + [retrieve(p) for p in expand(query)])
    exp_docs = [t[0] for t in exp]
    rows["Query expansion"].append(recall_at_k(exp_docs, rel))

    _, hyd = hyde(query)
    hyd_docs = [t[0] for t in hyd]
    rows["HyDE"].append(recall_at_k(hyd_docs, rel))

n = len(queries)
print(f"{'Подход':<18}{'recall@5':>10}")
for name in rows:
  print(f"{name:<18}{np.mean(rows[name]):>10.3f}")

Подход              recall@5
Обычный поиск          0.634
Query expansion        0.661
HyDE                   0.830


In [13]:
def basic_search(query):
  ret = retrieve_dict(query)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids

In [14]:
def precision_at_k(found_ids, relevant_ids, k):
    """Доля релевантных среди k выданных чанков."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / k


def recall_at_k(found_ids, relevant_ids, k):
    """Доля релевантных чанков, попавших в top-k."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / len(relevant_ids)


def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0


K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"])
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Basic Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.20   1.00   1.00
как подключить саундбар по HDMI ARC            1   0.00   0.00   0.00
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.20   1.00   1.00
сколько дней длится пробный период             1   0.40   2.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   2.00   0.50
на каком тарифе доступен звук 5.1              2   0.20   0.50   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
ско

In [16]:
def retrieve_hyde_dict(query, answer, k=5, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    ) 
  return out


def hyde_search(query, top_k=5):
  """HyDE: ищем по среднему векторов запроса и гипотетического ответа."""
  doc = generate([
      {"role": "system", "content": "Ты пишешь короткие ответы из базы знаний "
                                    "онлайн-кинотеатра."},
      {"role": "user", "content": f"Напиши короткий (1-2 предложения) фрагмент-"
        f"ответ из базы знаний на вопрос клиента официальным языком. Только "
        f"текст ответа.\n\nВопрос: {query}"}], max_tokens=110)
  ret = retrieve_hyde_dict(query, doc, top_k)
  doc_ids = [i['doc_id'] for i in ret]
  # print(doc_ids)
  return doc_ids

# found = dense_search('боюсь, что ребёнок случайно купит фильм с моей карты')
# print(found)

K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = hyde_search(q["query"])
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(list(set(found)), rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> HyDE Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> HyDE Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.00   0.00   0.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.20   1.00   1.00
как подключить саундбар по HDMI ARC            1   0.20   1.00   0.20
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.40   1.00   1.00
сколько дней длится пробный период             1   0.40   1.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   1.00   1.00
на каком тарифе доступен звук 5.1              2   0.40   1.00   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
скол

In [ ]:
def expanded_search(query, top_k=5):
  paras = expand(query)
  found = [retrieve_dict(p, k=5) for p in paras]
  # print('>>> found: ', found)

  qs = [basic]
  for item in found:
    if item not in qs:
        qs.append(item)
  expanded = rrf(qs)

  doc = generate([
      {"role": "system", "content": "Ты пишешь короткие ответы из базы знаний "
                                    "онлайн-кинотеатра."},
      {"role": "user", "content": f"Напиши короткий (1-2 предложения) фрагмент-"
        f"ответ из базы знаний на вопрос клиента официальным языком. Только "
        f"текст ответа.\n\nВопрос: {query}"}], max_tokens=110)
  ret = retrieve_hyde_dict(query, doc, top_k)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids


K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = expanded_search(q["query"])
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(list(set(found)), rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Expanded Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

# Stage 3 - Report
- transformation is needed: Query expansion